# Decoupled Delta-Net: Google Colab Runner
**Project:** Sub-quadratic linear recurrence with surprise-triggered error reservoir
**Author:** Jaime Jarauta
**Pre-Registered Question:** Can decoupled delta recurrence maintain associative recall at > 150 tok/s on commodity GPUs/CPUs?

In [ ]:
# Step 1: Hardware Pre-flight & Credit Guard
import sys, os, time
import torch
from IPython import get_ipython

def auto_kill_on_error(self, etype, value, tb, tb_offset=None):
    self.showtraceback((etype, value, tb), tb_offset=tb_offset)
    print("\n[Credit Guard Alert] Unhandled error caught! Unassigning Colab GPU immediately...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception:
        pass

ip = get_ipython()
if ip is not None:
    ip.set_custom_exc((Exception,), auto_kill_on_error)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")


In [ ]:
# Step 2: Dependencies & Ingestion
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.exists("decoupled-delta-net"):
        os.system("git clone -q -b run https://github.com/jaimejarauta/decoupled-delta-net.git")
    if "decoupled-delta-net/src" not in sys.path:
        sys.path.insert(0, "decoupled-delta-net/src")

if "src" not in sys.path:
    sys.path.insert(0, "src")

from decoupled_delta.model import DecoupledDeltaModel
from decoupled_delta.metrics import benchmark_associative_throughput
print("[Ready] Decoupled Delta-Net loaded successfully.")


In [ ]:
# Step 3: Run Smoke Benchmark
model = DecoupledDeltaModel(vocab_size=256, d_model=64, d_k=32, d_v=32, num_layers=2).to(device)
metrics = benchmark_associative_throughput(model, seq_len=1024, batch_size=2, device=device)
print("=" * 60)
print(f"Throughput : {metrics['tokens_per_sec']:.1f} tok/s (Target: >= 150 tok/s)")
print(f"Per-Token Latency : {metrics['ms_per_token']:.3f} ms/token")
print(f"Surprise Rate : {metrics['surprise_rate'] * 100:.1f}%")
gate_passed = metrics['tokens_per_sec'] >= 150 and metrics['ms_per_token'] <= 10.0
print(f"Gate 1 Status: {'PASSED' if gate_passed else 'FAILED'}")
print("=" * 60)


In [ ]:
# Step 4: Auto-Kill Runtime on Completion to Save Colab Credits
AUTO_KILL_ON_FINISH = True # @param {type:"boolean"}
if AUTO_KILL_ON_FINISH:
    print("\n[Credit Guard] Run complete. Unassigning runtime immediately to save credits...")
    try:
        from google.colab import runtime
        runtime.unassign()
    except Exception as e:
        print(f"(Skipped auto-kill: {e})")
